# Easy Italian News
- download and clean up episodes
- download mp3 files as well
- this is good for 1 month (can change date to do it for other months)

# Find all days with news

In [1]:
import requests
from bs4 import BeautifulSoup
import re

l_to_remove = [#r'\n+', 
    r'www\..*\n',
    r'Click the link..*\n', 
    r'https://..*\n', 
    r'\xa0\n', 
    r'Creative Commons..*\n',
    r'Image.*\n', 
    # r'it\.[\w+]..*\n', 'tg24\..*\n',
    # r'[\s]Jeffrey Zeldman\n',
    # r'[\s]Zdravko Petrov\n', 
    # r'[\s]Chris Watt\n',
    # r'[\s]GovernmentZA.*\n',
    # r'[\s]Attribution..*\n',
    # r'[\s]Elvert Barnes..*\n',
    # r'[\s]Maritza Ríos..*\n', 
    # r'[\s]si.robi.*\n',
    # r'[\s]Steven Depolo.*\n',
    # r'[\s]Brendan Keene.*\n',
    # r'[\s]Francesco Ranieri.*\n',
    # r'[\s]Nathan Keirn.*\n',
    # r'[\s]Giorgio Minguzzi.*\n',
    #r'^[\w+]\.[\w+]\.[\w+]$'
    #r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    
    r' \n',    
]

def merge_paragraphs(text):
    # Split on one or more empty lines
    blocks = re.split(r'\n\s*\n+', text.strip())

    paragraphs = []
    for block in blocks:
        # Remove leading/trailing whitespace from each line
        lines = [line.strip() for line in block.splitlines() if line.strip()]

        # Join lines within the block into a single paragraph
        paragraph = " ".join(lines)

        if paragraph:
            paragraphs.append(paragraph)

    return paragraphs

# Pattern to remove 2-4 words separated by dots.
# The only space can be at the beginnign of the line
pattern = re.compile(
    r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    re.MULTILINE
)

In [52]:
# Specify the URL of the website you want to scrape
url = 'https://easyitaliannews.com/2025/06/'

# To avoid server error: 403
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/117.0"
}

# Send a GET request to the URL
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    l_tds = soup.find_all('td')

    l_of_news = []
    for td in soup.find_all('td'):
        try:
            l_of_news.append(td.find('a').get('href'))
        except:
            pass

# %%time
# l_of_news

# len(l_of_news)

In [53]:
%%time
pattern = re.compile(
    r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    re.MULTILINE
)

for url in l_of_news:
# for url in l_of_news[:1]:
    news_date = url[28:38].replace('/', '-')
    print(news_date)

    # Send a GET request to the URL
    response = requests.get(url, headers=headers)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        for h4 in soup.find_all("h4"):
            p = soup.new_tag("p")
            p.string = "<b>" + h4.get_text(strip=True) + "</b>"
        
            h4.replace_with(p)
            
            # Add two line breaks after the paragraph
            p.insert_after(soup.new_tag("br"))
            p.insert_after(soup.new_tag("br"))

        ns = soup.find('div', {'class': 'entry-content'})
            
        # remove <strong> tags
        for strong in ns.find_all("strong"):
            strong.unwrap()

        # Remove <div class="wp-caption alignnone">...</div>
        for div in ns.select("div.wp-caption.alignnone"):
            div.decompose()
            
        # Remove all <a> tags and their contents
        for a in ns.find_all("a"):
            a.decompose()

        # Remove all <img> tags and their contents
        for img in ns.find_all("img"):
            img.decompose()

        # Remove all tags after "Subscribe" - which is a link
        # So, I remove from the next tag!
        for p in ns.find_all("p"):
            if "to receive each bulletin via email (free!)" in p.get_text():
                # remove this paragraph and everything after it
                current = p
                while current:
                    nxt = current.find_next_sibling()
                    current.decompose()
                    current = nxt
                break

        # Extracting all paragraphs
        l_content = []
        #mp3 = ''
        paragraphs = ns.find_all(True)
        for idx, paragraph in enumerate(paragraphs):
            # print(f"Paragraph {idx+1}: {paragraph.text}")
            p = str(paragraph.text)
            l_content.append(p)


        s = ('\n').join(l_content).split('Il tuo aiuto per noi è importante!')[0].split('Subscribe')[0]


        t = ''
        for el in l_to_remove:
            t = re.sub(el, '\n', s)
            s = t

        # # Remove the links to websites (2-4 words separate by dots)
        # cleaned_text = pattern.sub('', s)

        # Group text by paragraphs
        #paragraphed = merge_paragraphs(cleaned_text)
        paragraphed = merge_paragraphs(s)

        # Output with blank line before paragraphs that start with a capitalized word
        result = []
        for i, para in enumerate(paragraphed):
            if re.fullmatch(r'[A-Z]+', para.split()[0]):
                result.append("")  # blank line
            result.append(para)
        
        final_text = "\n\n".join(result)
        
        
        with open(f'EasyItalianNews_{news_date}.txt', 'w', encoding='utf-8') as f:
            f.write(final_text)

        # Find mp3 file
        audio_source = soup.select_one("audio source")
        if audio_source:
            mp3_url = audio_source.get("src").split('?')[0]

        doc = requests.get(mp3_url)
    
        with open(f'EasyItalianNews_{news_date}.mp3', 'wb') as f:
            f.write(doc.content)
        
print('done')

2025-06-03


2025-06-05


2025-06-07


2025-06-10


2025-06-12


...

2025-06-21


2025-06-24


2025-06-26


2025-06-28


done
CPU times: total: 15.8 s
Wall time: 1min 36s


# Yearly

In [62]:
import pandas as pd
# start_date = pd.to_datetime('2018-09-01').strftime('%Y-%m')
start_date = pd.to_datetime('2018-11-01').strftime('%Y-%m')
# start_date = pd.to_datetime('2026-06-01').strftime('%Y-%m')
l_year_months = pd.period_range(start_date, freq='M', periods=80).strftime('%Y/%m').tolist()
l_year_months

['2018/11',
 '2018/12',
 '2019/01',
 '2019/02',
 '2019/03',
 '2019/04',
 '2019/05',
 '2019/06',
 '2019/07',
 '2019/08',
 '2019/09',
 '2019/10',
 '2019/11',
 '2019/12',
 '2020/01',
 '2020/02',
 '2020/03',
 '2020/04',
 '2020/05',
 '2020/06',
 '2020/07',
 '2020/08',
 '2020/09',
 '2020/10',
 '2020/11',
 '2020/12',
 '2021/01',
 '2021/02',
 '2021/03',
 '2021/04',
 '2021/05',
 '2021/06',
 '2021/07',
 '2021/08',
 '2021/09',
 '2021/10',
 '2021/11',
 '2021/12',
 '2022/01',
 '2022/02',
 '2022/03',
 '2022/04',
 '2022/05',
 '2022/06',
 '2022/07',
 '2022/08',
 '2022/09',
 '2022/10',
 '2022/11',
 '2022/12',
 '2023/01',
 '2023/02',
 '2023/03',
 '2023/04',
 '2023/05',
 '2023/06',
 '2023/07',
 '2023/08',
 '2023/09',
 '2023/10',
 '2023/11',
 '2023/12',
 '2024/01',
 '2024/02',
 '2024/03',
 '2024/04',
 '2024/05',
 '2024/06',
 '2024/07',
 '2024/08',
 '2024/09',
 '2024/10',
 '2024/11',
 '2024/12',
 '2025/01',
 '2025/02',
 '2025/03',
 '2025/04',
 '2025/05',
 '2025/06']

In [63]:
for year_month in l_year_months:
    

    # Specify the URL of the website you want to scrape
    url = f'https://easyitaliannews.com/{year_month}/'
    # # Specify the URL of the website you want to scrape
    # url = 'https://easyitaliannews.com/2025/06/'
    
    # To avoid server error: 403
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/117.0"
    }
    
    # Send a GET request to the URL
    response = requests.get(url, headers=headers)
    
    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
    
        l_tds = soup.find_all('td')
    
        l_of_news = []
        for td in soup.find_all('td'):
            try:
                l_of_news.append(td.find('a').get('href'))
            except:
                pass
    
    # %%time
    # print(l_of_news)
    
    # len(l_of_news)
    for url in l_of_news:
    # for url in l_of_news[:1]:
        news_date = url[28:38].replace('/', '-')
        print(news_date, end=", ")
    
        # Send a GET request to the URL
        response = requests.get(url, headers=headers)
    
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            # Parse the HTML content using BeautifulSoup
            soup = BeautifulSoup(response.text, 'html.parser')
    
            for h4 in soup.find_all("h4"):
                p = soup.new_tag("p")
                p.string = "<b>" + h4.get_text(strip=True) + "</b>"
            
                h4.replace_with(p)
                
                # Add two line breaks after the paragraph
                p.insert_after(soup.new_tag("br"))
                p.insert_after(soup.new_tag("br"))
    
            ns = soup.find('div', {'class': 'entry-content'})
                
            # remove <strong> tags
            for strong in ns.find_all("strong"):
                strong.unwrap()
    
            # Remove <div class="wp-caption alignnone">...</div>
            for div in ns.select("div.wp-caption.alignnone"):
                div.decompose()
                
            # Remove all <a> tags and their contents
            for a in ns.find_all("a"):
                a.decompose()
    
            # Remove all <img> tags and their contents
            for img in ns.find_all("img"):
                img.decompose()
    
            # Remove all tags after "Subscribe" - which is a link
            # So, I remove from the next tag!
            for p in ns.find_all("p"):
                if "to receive each bulletin via email (free!)" in p.get_text():
                    # remove this paragraph and everything after it
                    current = p
                    while current:
                        nxt = current.find_next_sibling()
                        current.decompose()
                        current = nxt
                    break
    
            # Extracting all paragraphs
            l_content = []
            #mp3 = ''
            paragraphs = ns.find_all(True)
            for idx, paragraph in enumerate(paragraphs):
                # print(f"Paragraph {idx+1}: {paragraph.text}")
                p = str(paragraph.text)
                l_content.append(p)
    
    
            s = ('\n').join(l_content).split('Il tuo aiuto per noi è importante!')[0].split('Subscribe')[0]
    
    
            t = ''
            for el in l_to_remove:
                t = re.sub(el, '\n', s)
                s = t
    
            # # Remove the links to websites (2-4 words separate by dots)
            # cleaned_text = pattern.sub('', s)
    
            # Group text by paragraphs
            #paragraphed = merge_paragraphs(cleaned_text)
            paragraphed = merge_paragraphs(s)
    
            # Output with blank line before paragraphs that start with a capitalized word
            result = []
            for i, para in enumerate(paragraphed):
                if re.fullmatch(r'[A-Z]+', para.split()[0]):
                    result.append("")  # blank line
                result.append(para)
            
            final_text = "\n\n".join(result)
            
            
            with open(f'EasyItalianNews_{news_date}.txt', 'w', encoding='utf-8') as f:
                f.write(final_text)
    
            # Find mp3 file
            audio_source = soup.select_one("audio source")
            if audio_source:
                mp3_url = audio_source.get("src").split('?')[0]
    
            doc = requests.get(mp3_url)
        
            with open(f'EasyItalianNews_{news_date}.mp3', 'wb') as f:
                f.write(doc.content)
        
print('done')

2018-11-03, 

2018-11-06, 

2018-11-08, 

2018-11-10, 

2018-11-13, 

...

2025-06-21, 

2025-06-24, 

2025-06-26, 

2025-06-28, 

done


In [5]:
result[-5:]

['La vittoria corona una stagione eccezionale, segnata da una clamorosa inversione di tendenza nei playoff.',
 'Infatti, dopo aver perso due delle prime tre partite, i Knicks ne hanno vinte 15 delle successive 16.',
 'Il grande successo è dovuto anche al nuovo allenatore della squadra Mike Brown, capace di rinforzare il gruppo con un gioco rapido e un maggiore utilizzo delle rotazioni.',
 'Nel team vincente c’è anche un pezzo d’Italia grazie all’assistente allenatore Riccardo Fois.',
 'Per New York, da sempre capitale culturale del basket, è finita l’epoca delle delusioni e inizia un periodo di grandi festeggiamenti.']

In [8]:
# Copying text to clipboard

In [22]:
import pyperclip

pyperclip.copy(str(final_text))
print("Copied to clipboard")

Copied to clipboard


In [9]:
# Finding tags

In [26]:
for tag in soup.find_all(True):
    txt = tag.get_text(" ", strip=True)
    if "mp3" in txt or "Emergency" in txt or "dreese" in txt:
        print(tag.name)
        print(tag)
        print("-" * 80)